In [1]:
!pip install selenium webdriver-manager

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
import random
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
chrome_options = Options()
chrome_options.binary_location = r"C:\Program Files (x86)\chrome-win64\chrome.exe"
chrome_options.add_argument(r"--user-data-dir=C:\ChromeUserData")

service = Service(ChromeDriverManager(driver_version="142.0.7444.175").install())
driver = webdriver.Chrome(service=service, options=chrome_options)

In [4]:
def get_melon_lyrics(song_title):
    wait = WebDriverWait(driver, 10)
    
    try:
        try:
            search_box = driver.find_element(By.ID, "top_search")
        except:
            driver.get("https://www.melon.com/")
            search_box = wait.until(EC.presence_of_element_located((By.ID, "top_search")))

        search_box.clear() 
        search_box.send_keys(song_title)
        search_box.send_keys(Keys.ENTER)

        lyrics_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.btn.btn_icon_detail")))
        lyrics_btn.click()

        lyrics_area = wait.until(EC.presence_of_element_located((By.ID, "d_video_summary")))
        
        time.sleep(0.5) 
        lyrics_text = lyrics_area.text
        
        print(f"✅ [{song_title}] 수집 성공!")

        driver.back()
        return lyrics_text

    except Exception as e:
        try:
            driver.back()
        except:
            pass # 뒤로가기도 안 되면 그냥 넘김 (다음 곡 시작 때 메인으로 감)
        return None

In [5]:
def get_top100_titles():
    driver.get("https://www.melon.com/chart/index.htm")
    time.sleep(2)
    
    # 차트에 있는 모든 곡 제목 요소 찾기
    song_elements = driver.find_elements(By.CSS_SELECTOR, ".ellipsis.rank01 a")
    artist_elements = driver.find_elements(By.CSS_SELECTOR, ".ellipsis.rank02 span a") 
    
    titles = []
    for s, a in zip(song_elements, artist_elements):
        titles.append(f"{a.text} {s.text}") # "가수 제목" 형태로 저장
    
    return titles

In [6]:
top100_list = get_top100_titles()
print(f"📊 현재 차트에서 {len(top100_list)}곡을 확인했습니다.")

random.shuffle(top100_list)

all_data = []
current_index = 0

print("🚀 가사 수집을 시작합니다. (목표: 5곡)")

while len(all_data) < 5 and current_index < len(top100_list):
    song = top100_list[current_index]
    
    lyrics = get_melon_lyrics(song)
    
    if lyrics:
        print(f"✅ [{len(all_data)+1}/5] '{song}' 수집 성공")
        all_data.append({"title": song, "lyrics": lyrics})
    else:
        print(f"⚠️ [{song}] 수집 건너뜀 (가사 없음 또는 페이지 오류)")
    
    current_index += 1
    time.sleep(1)

print("\n✨ 모든 수집 절차가 완료되었습니다.")

📊 현재 차트에서 100곡을 확인했습니다.
🚀 가사 수집을 시작합니다. (목표: 5곡)
✅ [ 하얀 그리움] 수집 성공!
✅ [1/5] ' 하얀 그리움' 수집 성공
✅ [ 사랑인가 봐] 수집 성공!
✅ [2/5] ' 사랑인가 봐' 수집 성공
✅ [ 가까운 듯 먼 그대여] 수집 성공!
✅ [3/5] ' 가까운 듯 먼 그대여' 수집 성공
✅ [ body] 수집 성공!
✅ [4/5] ' body' 수집 성공
⚠️ [ 모든 날, 모든 순간 (Every day, Every Moment)] 수집 건너뜀 (가사 없음 또는 페이지 오류)
⚠️ [ Love Me More] 수집 건너뜀 (가사 없음 또는 페이지 오류)
⚠️ [ 고민중독] 수집 건너뜀 (가사 없음 또는 페이지 오류)
✅ [ DRIP] 수집 성공!
✅ [5/5] ' DRIP' 수집 성공

✨ 모든 수집 절차가 완료되었습니다.


In [7]:
!pip install langchain-groq

In [10]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import os

os.environ["GROQ_API_KEY"] = "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" 

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """
    당신은 음악 가사 분석 전문가입니다. 
    제공된 가사를 분석하여 반드시 아래의 JSON 형식으로만 답변하세요. 다른 설명은 하지 마세요.
    
    {{
        "mood": "가사의 전반적인 분위기 (예: 슬픔, 아련, 기쁨)",
        "sentiment_score": 0.0에서 1.0 사이의 숫자 (1에 가까울수록 슬픔이 강함),
        "visual_keyword": "가사에서 연상되는 한 가지 시각적 배경"
    }}

    가사: {lyrics}
    """
)

chain = prompt | llm | JsonOutputParser()

def analyze_lyrics_with_ai(lyrics):
    try:
        result = chain.invoke({"lyrics": lyrics[:500]})
        return result
    except Exception as e:
        print(f"분석 에러: {e}")
        return {"mood": "Error", "sentiment_score": 0, "visual_keyword": "Error"}

df=pd.DataFrame(all_data)
print("--- Groq AI 기반 가사 분석 시작 ---")
df['ai_analysis'] = df['lyrics'].apply(analyze_lyrics_with_ai)

df_final = pd.concat([df, pd.json_normalize(df['ai_analysis'])], axis=1).drop(columns=['ai_analysis'])

display(df_final)

--- Groq AI 기반 가사 분석 시작 ---


,title,lyrics,mood,sentiment_score,visual_keyword
0,하얀 그리움,하얀 눈이 내려와\n내 맘을 아프게 해\n날 힘들게 해\n\n사라져 버린 눈처럼\n...,슬픔,0.8,하얀 눈
1,사랑인가 봐,너와 함께 하고 싶은 일들을\n상상하는 게\n요즘 내 일상이 되고\n너의 즐거워하는...,아련,0.8,달빛 아래서 함께 걷는 밤
2,가까운 듯 먼 그대여,저 달빛에 그려지는\n그대의 미소를 간직해\n그을진 저 노을 속에\n그대 얼굴이 떠...,아련,0.8,밤하늘
3,body,Give me love and I’ma give it right back\nWho ...,기쁨,0.2,마스
4,DRIP,When I dress I don’t think so much\nI could be...,자신감과 기쁨,0.2,밤하늘의 별들


In [11]:
!pip install snowflake-connector-python snowflake-sqlalchemy pandas

In [12]:
import snowflake.connector
from uuid import uuid4

ctx = snowflake.connector.connect(
    user='USIM2214',
    password='Tladntjr221412@',
    account='cbhavmt-bq90093', 
    warehouse='COMPUTE_WH',
    database='LYRIC_DB',
    schema='PUBLIC'
)

df_final.columns = [col.upper() for col in df_final.columns]
                   
cursor = ctx.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS LYRIC_DB")
cursor.execute("USE DATABASE LYRIC_DB")
cursor.execute("""
    CREATE OR REPLACE TABLE LYRIC_ANALYSIS (
        TITLE STRING,
        LYRICS STRING,
        MOOD STRING,
        SENTIMENT_SCORE FLOAT,
        VISUAL_KEYWORD STRING,
        CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )
""")

from snowflake.connector.pandas_tools import write_pandas

success, nchunks, nrows, _ = write_pandas(ctx, df_final, 'LYRIC_ANALYSIS')

if success:
    print(f"✅ 총 {nrows}개의 데이터가 Snowflake에 성공적으로 적재되었습니다!")
else:
    print("❌ 적재 실패")

cursor.close()
ctx.close()

✅ 총 5개의 데이터가 Snowflake에 성공적으로 적재되었습니다!


In [13]:
!pip install fastapi uvicorn nest-asyncio

In [1]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
import webbrowser 
from threading import Timer 
from fastapi.responses import HTMLResponse

nest_asyncio.apply()
app = FastAPI()

@app.get("/")
def read_root():
    return {"message": "서버가 정상 작동 중입니다. /lyrics 주소로 이동하세요!"}

@app.get("/lyrics", response_class=HTMLResponse)
def get_all_lyrics():
    html_table = df_final.to_html(index=False, justify='center', border=1)

    html_content = f"""
    <html>
        <head>
            <meta charset="utf-8">
            <title>멜론 가사 AI 분석 결과</title>
            <style>
                body {{ font-family: sans-serif; padding: 20px; background-color: #f8f9fa; }}
                h2 {{ color: #2c3e50; text-align: center; }}
                table {{ width: 100%; border-collapse: collapse; background: white; margin-top: 20px; }}
                th {{ background-color: #27ae60; color: white; padding: 12px; }}
                td {{ padding: 10px; border: 1px solid #ddd; text-align: center; }}
                tr:nth-child(even) {{ background-color: #f2f2f2; }}
                tr:hover {{ background-color: #e9ecef; }}
            </style>
        </head>
        <body>
            <h2>📊 멜론 차트 가사 AI 분석 결과 (Top 5)</h2>
            {html_table}
        </body>
    </html>
    """
    return html_content

def open_browser():
    webbrowser.open("http://127.0.0.1:8000/lyrics")

if __name__ == "__main__":
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    
    Timer(1.5, open_browser).start()
    
    print("🚀 서버가 시작되었습니다. 잠시 후 자동으로 분석 결과창이 열립니다!")
    
    import asyncio
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())

🚀 서버가 시작되었습니다. 잠시 후 자동으로 분석 결과창이 열립니다!


INFO:     Started server process [20304]
INFO:     Waiting for application startup.
